In [ ]:
import random, os
import numpy as np
import torch
os.environ["CUDA_VISIBLE_DEVICES"]="0,2"

from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import pandas as pd
import re

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'

## Setting the seed

In [5]:
def set_seed(seed_value):
    # Set seed for reproducibility.
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED']=str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic=True    
    torch.backends.cudnn.benchmark=True
    torch.cuda.manual_seed_all(seed_value)

# Dataset

In [6]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load embeddings (evidence_eval)
embedd_test_path = f'{data_dir}/test/embedd_test_eval2.npy'
evidence_embeddings_eval = np.load(embedd_test_path)
print(evidence_embeddings_eval.shape)
evidence_embeddings_eval = torch.from_numpy(evidence_embeddings_eval).to(device1)

#load embeddings (evidence_title)
embedd_test_path = f'{data_dir}/test/embedd_test_eval_title.npy'
evidence_embeddings_title = np.load(embedd_test_path)
print(evidence_embeddings_title.shape)
evidence_embeddings_title = torch.from_numpy(evidence_embeddings_title).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test2.csv'
evidence_df = pd.read_csv(evidence_test_path)
print(len(evidence_df))

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test_eval.csv'
evidence_eval_df = pd.read_csv(evidence_test_path)
print(len(evidence_eval_df))

#load title evidence
evidence_test_path = f'{data_dir}/test/evidence_test_eval_title.csv'
evidence_eval_title_df = pd.read_csv(evidence_test_path)
print(len(evidence_eval_title_df))

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)
(21801, 4096)
(1897, 4096)
21586
21801
1897


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,aee000f3-d2b0-4de5-8206-a96e9c203207,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,b80ea7a2-5084-422f-94d1-e67e7e29819b,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,a9c1319b-ed69-4b1b-8486-05ad3d444e22,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,b07f0006-0628-49cf-b5b4-c0c7e88d3190,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,a9d31e99-6402-4a41-a4af-52de1aebeb16,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


## Load the retrieval

In [8]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.93s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

## Load the generative LLM

In [9]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
# cache_dir= '/raid/deallab/.cache')
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto',
    # cache_dir= '/raid/deallab/.cache'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:16<00:00,  4.18s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

## Measuring the recall

In [ ]:
def match_sample_id(query_id, doc_id):
    query_sample_id=qa_df.loc[query_id,'sample_id']
    doc_sample_id=evidence_eval_df.loc[doc_id,'sample_id']
    if query_sample_id==doc_sample_id:
        return 1
    else:
        return 0
    

In [16]:
def match_sample_id2(query_id, title):
    query_sample_id=qa_df.loc[query_id,'sample_id']
    doc_sample_id=evidence_eval_title_df.loc[evidence_eval_title_df['title']==title,'sample_id'].item()
    if query_sample_id==doc_sample_id:
        return 1
    else:
        return 0

In [ ]:
def cal_sample_title(query_id,documents):
    titles_of_documents = evidence_df[evidence_df['text'].isin(documents)]['title'].values
    query_sample_id=qa_df.loc[query_id,'sample_id']
    titles_of_query = set(evidence_eval_df[evidence_eval_df['sample_id']==query_sample_id]['title'].values)
    
    print(titles_of_documents)
    print(titles_of_query)
    return len(set(titles_of_documents) & titles_of_query)/len(titles_of_query)

## Retrieval Funcs

In [17]:
# retrive docs from the document embeddings
def retrieve_documents(query,num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
    idx=[idx for idx in top_results if idx < len(evidence_df)]
    return res, idx

In [18]:
# retrive docs from the document embeddings
def retrieve_documents_eval(query,num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings_eval)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence_eval_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_eval_df)]
    idx=[idx for idx in top_results if idx < len(evidence_eval_df)]
    return res, idx

In [19]:
# retrive docs from the document embeddings
def retrieve_documents2(query, embed, evidence, num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, embed)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence.loc[idx, 'text'] for idx in top_results if idx < len(evidence)]
    titles=[evidence.loc[idx, 'title'] for idx in top_results if idx < len(evidence)]
    # idx=[idx for idx in top_results if idx < len(evidence)]
    return res, titles

In [20]:
# retrive docs from the document embeddings
def retrieve_documents3(query, embed, evidence, num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, embed)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence.loc[idx, 'text'] for idx in top_results if idx < len(evidence)]
    # titles=[evidence.loc[idx, 'title'] for idx in top_results if idx < len(evidence)]
    idx=[idx for idx in top_results if idx < len(evidence)]
    return res, idx

In [21]:
# retrive docs from the document embeddings
def retrieve_titles(query,num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings_title)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence_eval_title_df.loc[idx, 'title'] for idx in top_results if idx < len(evidence_eval_title_df)]
    idx=[idx for idx in top_results if idx < len(evidence_eval_title_df)]
    return res, idx

In [ ]:
def find_docs(titles):
    docs=[]
    for title in titles:
        docs.extend(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'])
    return docs

# Prompt Methods

In [27]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [34]:
def total_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    1. The query is an ambiguous question.
    2. Therefore, you must include the contents according to the various interpretations of the query in one answer by utilizing the given context.
    3. Each content according to the various interpretations of the query must be explained in one or two sentences.
    4. The total answer must be 5 sentences or less.
    Do not comment your answer and strictly follow this instructions.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    print(prompt)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [ ]:
def CoT(query, context):
    prompt = """
    Context information is below.
    ——————————
    {0}
    ——————————
    In context, there may be content A, B, C, etc. for an ambiguous question. 
    Let's create a long answer that includes various related content such as A, B, C, etc.
    Query: {1}
    Answer: Let's think about it step by step.
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

# Apply the Prompts to Basic RAG

In [35]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20
retrival_list=[]
scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs, doc_ids = retrieve_documents(query,10)
    
    tmp=0
    for doc_id in doc_ids:    
        tmp+=match_sample_id(idx, doc_id)
    res=tmp/len(retrieved_docs)
    
    print(" Retrival Match Rate:", res)
    dic=dict()
    dic['first_doc_retrival']=res
    
    title_retival_rate=cal_sample_title(idx,retrieved_docs)
    print("Title Match Rate:", title_retival_rate)
    dic['first_title_retrival']=title_retival_rate
    
    ans=total_answer(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate([ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
    retrival_list.append(dic)
    retrival_df=pd.DataFrame(retrival_list)
    print(dic)
        
scores_df=pd.DataFrame(scores_list)
print(scores_df.mean())

retrival_df=pd.DataFrame(retrival_list)
print(retrival_df.mean())

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/c50d55f43bde7e6a18e0eaa15a62fd63a930f1a1/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


 Retrival Match Rate: 0.5
['List of FIFA World Cup records and statistics'
 'List of footballers with more than 50 international goals'
 'List of footballers with 500 or more goals'
 'List of footballers with 500 or more goals'
 'List of footballers with 500 or more goals'
 "List of top international men's association football goal scorers by ..."
 "List of men's footballers with 50 or more international goals"
 'Football records and statistics in Spain'
 'Germany at the FIFA World Cup' 'FIFA World Cup top goalscorers']
{'List of FIFA World Cup records and statistics', 'List of footballers with 500 or more goals', 'International Federation of Football History & Statistics', "List of women's footballers with 100 or more international goals ...", 'List of footballers with more than 50 international goals'}
Title Match Rate: 0.6

    Context information is below.
    ---------------------
    Document: List of footballers with 500 or more goals



In top-level  football, 28 players have s

  0%|          | 0/20 [00:12<?, ?it/s]


KeyboardInterrupt: 

# Performance Comparsion

In [ ]:
# 기본 라그 예전 데이터 문서 10개 검색 함수는 기본함수 answer
# Basic RAG with 10 docs and prompt func is answer
# first_retrival    0.675
# first_title_retrival    0.9175
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-answer-len100_results.csv')
sf=sf[sf['length']<1000][:20]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      26.424077
length         31.200000
str_em         49.166667
Disambig-F1    44.894841
dtype: float64
34.44277482138087


In [ ]:
# 기본 라그 예전 데이터 문서 10개 검색 함수는 기본함수 toal_answer
# Basic RAG with 10 docs and prompt func is total_answer
# first_retrival    0.675
# first_title_retrival    0.9175
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG-total-seed24-len20_results.csv')
sf=sf[sf['length']<1000][:20]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
Unnamed: 0      9.500000
rougeLsum      38.582059
length         98.850000
str_em         61.666667
Disambig-F1    49.227564
dtype: float64
43.58096816265843


### Testing the prompts

In [ ]:
def answer_2020(query, title, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer this ambiuous question.
    1. You should include in answer the content according to the various interpretations of the question
    2. When using time-related information to answer the question, only use information before February 1, 2020.
    3. If you can’t answer the question, say "Not relevant" only.
    4. The answer should be made considering the title.
    5. Each contents should include the subject, verb, predicate, object, time, and place appropriately.
    6. Answers to questions should be no longer than 3 sentences.
    Do not comment your answer and strictly follow this instructions.
    Query: {1}
    Title: {2}
    Answer:
    """.format('\n'.join(context), query, title)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [ ]:
def total_answer_2020(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer this ambiuous question.
    1. Summarize all information in less than 100 characters. 
    2. You must include subject, verb, object, time, and place.
    Do not comment your answer and strictly follow this instructions.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

## Title RAG -> Retrieved title first and make tmp answers using chunks of same title.

In [ ]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20
retrival_list=[]
scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']   
    
    retrieved_docs, doc_ids = retrieve_documents(query,10)
    ori_ans=total_answer(query, retrieved_docs)
    print('Basic ans:',ori_ans)
    
    titles,title_ids=retrieve_titles(query, 10)
    docs=[]
    first_ans=[]
    
    first_ans.append(ori_ans)
    
    for title in titles:
        text_docs=evidence_df.loc[evidence_df['title']==title,'text'].to_list()
        docs_tensor=evidence_embeddings[evidence_df['title']==title]
        title_retrieved_docs,_=retrieve_documents3(query, docs_tensor, pd.DataFrame(text_docs, columns=['text']), 10)
        while 1:
            ans=answer_2020(query,title,title_retrieved_docs)
            if "Context information is below." not in ans:
                break
        print(ans)
        if "Not relevant" in ans:
            continue
        first_ans.append(ans)
    print(first_ans)
    
    final_ans=total_answer(query,first_ans)
    print('Final ans:', ori_ans+final_ans)
    scores=evaluate([final_ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

Basic ans: Ali Daei holds the record for the highest number of international goals with 109 goals for Iran, surpassing Ferenc Puskás' record of 84 goals. Cristiano Ronaldo has scored 1030 goals in his career, the highest among active players, with a total of 738 goals in his international career. Miroslav Klose holds the record for most World Cup goals with 16 goals, while Gerd Müller used to be the holder of that record from 1974 until it was broken by Ronaldo in 2006. Pelé scored 77 international goals, the highest among players from outside Europe, and Jürgen Klinsmann scored 11 goals in the World Cup, the highest among players from Germany. The top goalscorers in the World Cup history are Ali Daei, Cristiano Ronaldo, Ferenc Puskás, Kunishige Kamamoto, Godfrey Chitalu, Hussein Saeed, and Zainal Abidin, among others.
Ali Daei of Iran holds the record for the highest number of international goals with 109 goals, surpassing Ferenc Puskás of Hungary who had the record for 47 years. He a

  5%|▌         | 1/20 [04:40<1:28:54, 280.77s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.7461572885513306, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.19398072361946106, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.9859077334403992, 'start': 291, 'end': 309, 'answer': 'Christine Sinclair'}
{'rougeLsum': 41.70616113744076, 'length': 103.0, 'str_em': 100.0, 'Disambig-F1': 66.66666666666666}
Basic ans: The original artist of "Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Garfunkel. They recorded the song in March 1964, and it was initially released as a single in September 1965. The song was written by Paul Simon, and its origin

 10%|█         | 2/20 [08:12<1:11:58, 239.90s/it]

...

 95%|█████████▌| 19/20 [57:40<02:17, 137.72s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9980365037918091, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.9846311807632446, 'start': 190, 'end': 200, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9972240924835205, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.8490548133850098, 'start': 190, 'end': 200, 'answer': 'Ethan Peck'}
{'rougeLsum': 32.78688524590164, 'length': 82.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
Basic ans: The chief minister of Madh

100%|██████████| 20/20 [59:49<00:00, 179.47s/it]

{'score': 4.633776029550063e-08, 'start': 40, 'end': 50, 'answer': 'Kamal Nath'}
follow question : Who is the 15th chief minister of m. p?
short answer : ['Uma Bharti']
{'score': 3.4664896730873807e-08, 'start': 40, 'end': 50, 'answer': 'Kamal Nath'}
{'rougeLsum': 48.40764331210191, 'length': 96.0, 'str_em': 66.66666666666666, 'Disambig-F1': 0.0}


rougeLsum       35.940787
length         106.200000
str_em          64.166667
Disambig-F1     46.714286
dtype: float64